In [4]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
import os
import random

class MahimahiTraceManager:
    """
    Mengelola file trace Mahimahi (.log).
    Mengonversi timestamp (ms) menjadi Throughput (Mbps) dengan smoothing.
    """
    def __init__(self, folder_path="traces_folder"):
        self.traces = []
        # MTU standar 1500 bytes = 12000 bits
        PACKET_SIZE_BITS = 1500 * 8 
        
        if os.path.exists(folder_path):
            # Mengambil semua file .log dan mengurutkannya (0001 - 0008)
            files = [f for f in os.listdir(folder_path) if f.endswith('.log')]
            files.sort()
            
            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]
                        
                        if timestamps_ms:
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0
                            
                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    # Mbps = (jumlah paket * ukuran) / 1,000,000
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1
                            
                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)
                            
                            # Normalisasi: Jika trace terlalu kecil/besar, pastikan ada variasi
                            # Smoothing 3 detik agar agen melihat tren, bukan noise instan
                            smoothed = pd.Series(throughput_mbps).rolling(window=3, min_periods=1).mean().tolist()
                            
                            self.traces.append({
                                "name": file,
                                "data": [max(0.1, x) for x in smoothed] # Minimal 0.1 Mbps
                            })
                except Exception as e:
                    print(f"Gagal memproses file {file}: {e}")
        
        if not self.traces:
            print("⚠️ Folder traces_folder tidak ditemukan atau kosong!")
            self.traces.append({"name": "synth_fallback", "data": [10.0] * 500})

        self.active_trace = None
        self.ptr = 0

    def select_random_trace(self):
        """Digunakan saat pelatihan: pilih trace acak agar agen tidak overfit."""
        self.active_trace = random.choice(self.traces)
        self.ptr = random.randint(0, max(0, len(self.active_trace["data"]) - 110))
        return self.active_trace["name"]

    def set_trace_index(self, idx):
        """Digunakan saat evaluasi: pilih file spesifik 0-7."""
        self.active_trace = self.traces[idx % len(self.traces)]
        self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val

class FluidStreamingEnv(gym.Env):
    """
    Lingkungan RL dengan konsep Fluid QoE.
    Menghapus hukuman 'Tandon Air' < 10s agar agen lebih berani bereksplorasi.
    """
    def __init__(self, trace_manager):
        super(FluidStreamingEnv, self).__init__()
        self.trace_manager = trace_manager
        self.bitrates = [0.5, 2.5, 8.0] # Kualitas 0, 1, 2
        
        # State: [Buffer_Detik, Throughput_Mbps, Kualitas_Terakhir, Tren_Buffer, Tren_TP]
        self.observation_space = spaces.Box(
            low=np.array([0, 0, 0, -10, -20]),
            high=np.array([30, 50, 2, 10, 20]),
            dtype=np.float32
        )
        self.action_space = spaces.Discrete(3)
        self.state = None
        self.max_steps = 100
        self.current_step = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if options and "trace_idx" in options:
            trace_name = self.trace_manager.set_trace_index(options["trace_idx"])
        else:
            trace_name = self.trace_manager.select_random_trace()
            
        initial_tp = self.trace_manager.get_next_bandwidth()
        # Mulai dengan buffer 15 detik (kondisi aman)
        self.state = np.array([15.0, initial_tp, 1.0, 0.0, 0.0], dtype=np.float32)
        self.current_step = 0
        return self.state, {"trace": trace_name}

    def step(self, action):
        # Pastikan action adalah integer skalar
        if isinstance(action, np.ndarray):
            action = action.item()
        
        buffer, last_tp, last_action, _, _ = self.state
        chosen_bitrate = self.bitrates[int(action)]
        raw_tp = self.trace_manager.get_next_bandwidth()
        
        seg_duration = 5.0
        # Waktu download nyata berdasarkan bandwidth trace
        download_time = (chosen_bitrate * seg_duration / (raw_tp + 0.1))
        stalling = max(0, download_time - buffer)
        
        # Update level buffer (dikurangi waktu download, ditambah durasi segmen baru)
        new_buffer = max(0, buffer - download_time) + seg_duration
        new_buffer = min(new_buffer, 30.0)
        
        # Hitung Tren
        buf_trend = np.clip(new_buffer - buffer, -10, 10)
        tp_trend = np.clip(raw_tp - last_tp, -20, 20)
        
        # --- LOGIKA REWARD FLUID QoE ---
        # 1. Reward Bitrate (Ditingkatkan faktor 3.0 agar AI 'rakus' akan kualitas)
        reward = float(chosen_bitrate * 3.0)
        
        # 2. Penalti Stalling (Hanya dihukum jika buffer benar-benar habis)
        if stalling > 0:
            reward -= 250.0 
        
        # 3. Penalti Switching (Agar perpindahan tidak terlalu sering/flicker)
        reward -= float(abs(int(action) - int(last_action)) * 1.5)

        # 4. PENALTI TANDON AIR DIHAPUS
        # Agen sekarang bebas berada di buffer berapapun asalkan tidak macet.

        self.state = np.array([new_buffer, raw_tp, float(action), buf_trend, tp_trend], dtype=np.float32)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self.state, reward, done, False, {"raw_tp": raw_tp}

def run_experiment():
    # Setup Folder Output
    log_dir = "./rl_logs/"
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    
    # Inisialisasi
    tm = MahimahiTraceManager(folder_path="traces_folder/mahimahi")
    if not tm.traces:
        print("❌ Tidak ada file trace yang ditemukan.")
        return
        
    env = Monitor(FluidStreamingEnv(tm), log_dir)
    
    # Pelatihan PPO dengan Entropi Tinggi (0.10) untuk memaksa keberanian eksplorasi
    print("⏳ Memulai pelatihan agen 'Fluid-Intelligence' (300,000 langkah)...")
    model = PPO("MlpPolicy", env, verbose=1, 
                learning_rate=0.0003, 
                ent_coef=0.10, # Sangat tinggi agar berani mencoba High
                n_steps=2048,
                batch_size=128)
    
    model.learn(total_timesteps=300000)
    model.save("fluid_ndn_ai_model")
    print("✅ Pelatihan selesai. Model disimpan.")

    # Evaluasi 8 File Trace Secara Berurutan
    print("\n📈 Menghasilkan 8 Grafik Evaluasi Spesifik...")
    for i in range(len(tm.traces)):
        obs, info = env.reset(options={"trace_idx": i})
        trace_name = info["trace"]
        history = []
        
        for _ in range(100):
            # predict mengembalikan (action, states), action biasanya berupa numpy array
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            
            # Konversi action ke item skalar (int) agar tidak menyebabkan unhashable error di pandas
            action_val = action.item() if isinstance(action, np.ndarray) else int(action)
            
            history.append({
                'Throughput': info_step['raw_tp'],
                'Buffer': obs[0],
                'Action': action_val
            })
            
        df = pd.DataFrame(history)
        
        # Plotting
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Grafik Bitrate vs Throughput
        ax1.plot(df.index, df['Throughput'], label='Bandwidth Trace (Mbps)', color='blue', alpha=0.3)
        # Visual scale untuk Action (0=1Mbps, 1=4Mbps, 2=9Mbps) agar terlihat jelas di grafik
        visual_action = df['Action'].map({0: 1.0, 1: 4.0, 2: 9.0})
        ax1.step(df.index, visual_action, label='Keputusan Bitrate AI', color='red', linewidth=2.5, where='post')
        ax1.set_title(f"Evaluasi Trace: {trace_name}")
        ax1.set_ylabel("Mbps / Level")
        ax1.legend()
        ax1.grid(alpha=0.2)
        
        # Grafik Buffer
        ax2.fill_between(df.index, df['Buffer'], color='green', alpha=0.15, label='Level Isi Buffer')
        ax2.axhline(y=5, color='orange', linestyle='--', label='Garis Bahaya (5s)')
        ax2.set_ylabel("Detik")
        ax2.set_xlabel("Nomor Segmen")
        ax2.set_ylim(0, 35)
        ax2.legend()
        ax2.grid(alpha=0.2)
        
        plt.tight_layout()
        plot_filename = os.path.join(log_dir, f"result_{trace_name}.png")
        plt.savefig(plot_filename)
        plt.close()
        print(f"  > Gambar tersimpan: {plot_filename}")

    # Plot Progress Pelatihan
    try:
        x, y = ts2xy(load_results(log_dir), 'timesteps')
        plt.figure(figsize=(10, 5))
        plt.plot(x, pd.Series(y).rolling(window=100).mean(), color='purple')
        plt.title("Track Record Kemajuan Pelatihan (Reward)")
        plt.xlabel("Total Langkah")
        plt.ylabel("Rata-rata Skor")
        plt.savefig(os.path.join(log_dir, "training_progress.png"))
        plt.close()
        print(f"📈 Progres pelatihan disimpan: {os.path.join(log_dir, 'training_progress.png')}")
    except Exception as e:
        print(f"⚠️ Gagal menyimpan progress pelatihan: {e}")

if __name__ == "__main__":
    run_experiment()

⏳ Memulai pelatihan agen 'Fluid-Intelligence' (300,000 langkah)...
Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | 834      |
| time/              |          |
|    fps             | 5454     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | 950         |
| time/                   |             |
|    fps                  | 4341        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013328809 |
|    clip_fraction        | 0.216       |
|    clip_range           | 0.2      